# Day 1 실습 — ChatPromptTemplate·LCEL 체인과 프롬프트 설계

**목표**: 프롬프트 양식을 만들어 체인으로 연결하고, 역할·지시문·맥락·예시를 하나씩 더하며 답 품질 변화를 직접 확인한다.
**구성**: Part 1 첫 체인 완성·오류 다루기 → Part 2 설계 요소 실험(+다른 도메인) → Part 3 미니 프로젝트(+나만의 캐릭터 챗봇)

> **참고:** 실습 전 가상환경 활성화, `.env`의 OpenAI API 키, 패키지 설치(`uv sync`)를 확인한다.

## 0. 환경 준비·모델 생성

필요한 도구를 불러오고 모델·파서 객체를 만든다. 이 셀은 노트북 전체에서 한 번만 실행한다.

<details>
<summary>각 import의 역할</summary>

- `ChatOpenAI`: OpenAI 채팅 모델을 부르는 객체다.
- `ChatPromptTemplate`: 프롬프트 양식을 만드는 도구다.
- `StrOutputParser`: 답을 순수 문자열로 정리하는 파서다.
- `load_dotenv`: `.env`의 API 키를 불러온다.
</details>

In [15]:
import os
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini")
parser = StrOutputParser()
print("준비 완료:", llm.model_name)

준비 완료: gpt-4o-mini


## Part 1. 첫 체인 완성

완성 코드를 직접 쳐서 `prompt | llm | parser` 체인을 처음부터 만든다.

이 체인은 사실 **두 도메인을 오가는 번역 루프**다. 사람이 원하는 것(**사용자 도메인**)과 모델이 다음 글자를 예측하며 이어 쓰는 문서(**모델 도메인**)는 서로 다른 언어이고, 애플리케이션 개발자의 일은 이 둘 사이를 번역하는 것이다.

- `prompt`: 사용자 도메인 → 모델 도메인 (**순방향 번역**) — 질문을 "모델이 이어 쓰고 싶어지는 문서" 형태로 바꾼다
- `llm`: 모델 도메인 안에서의 완성 (다음 글자를 예측해 이어 쓴다)
- `parser`: 모델 도메인 → 사용자 도메인 (**역방향 번역**) — 모델의 텍스트 출력을 프로그램이 쓸 수 있는 형태로 되돌린다

아래 1-1~1-4에서 이 세 조각을 하나씩 만들어본다. Day03(Structured Output)은 이 **역방향 번역을 더 정교하게 만드는 장**이라고 볼 수 있다.

### 1-1. 모델 직접 호출

양식 없이 모델에 질문을 바로 보내 본다. 답은 **메시지 객체**로 온다 (아래 `type`으로 확인).

In [16]:
# TODO: llm.invoke(...)로 질문을 보내고 답을 받으세요
answer = llm.invoke("LangChain을 한 문장으로 설명해줘") # 제로샷

print(answer.content)
print("타입:", type(answer))   # 메시지 객체

LangChain은 다양한 언어 모델과 데이터 소스를 연결하여 자연어 처리(NLP) 애플리케이션을 쉽게 구축할 수 있도록 돕는 프레임워크입니다.
타입: <class 'langchain_core.messages.ai.AIMessage'>


### 1-2. ChatPromptTemplate·변수 바인딩

빈칸 `{topic}`이 있는 양식을 만들고, 값을 채워 어떤 메시지가 만들어지는지 확인한다.

In [18]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 비전공자에게 친절히 설명하는 강사다."),

    # TODO: human 메시지에 빈칸 {topic}을 넣어 질문을 완성하세요
    ("human", "{topic}를 한문장으로 쉽게 설명해줘"),
])

# TODO: 빈칸 {topic}에 "API"를 채워 실행하세요
messages = prompt.invoke({"topic":"Langchain"})

print(messages)

messages=[SystemMessage(content='너는 비전공자에게 친절히 설명하는 강사다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Langchain를 한문장으로 쉽게 설명해줘', additional_kwargs={}, response_metadata={})]


### 1-3. StrOutputParser로 답 정리

파서를 쓰기 전과 후의 **결과 타입 차이**를 눈으로 비교한다.

> **참고:** 파서 전은 메시지 객체, 파서 후는 바로 출력·저장할 수 있는 텍스트다 (langchain 1.x에서는 타입이 `TextAccessor`로 나오지만 문자열처럼 쓸 수 있다).

In [19]:
# raw = llm.invoke(prompt.invoke({"topic": "API"}))
raw = llm.invoke(messages)
print("파서 전 타입:", type(raw))

파서 전 타입: <class 'langchain_core.messages.ai.AIMessage'>


In [20]:
raw

AIMessage(content='Langchain은 AI 모델과 다양한 데이터 소스를 연결해 자연어 처리 작업을 쉽게 할 수 있게 도와주는 도구입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 38, 'total_tokens': 66, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f5d25cc737', 'id': 'chatcmpl-EMUGOSSRBry3seNJOWHsKWkvKd43H', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a08a64-c752-7920-8d9f-e7c686ec7c41-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 38, 'output_tokens': 28, 'total_tokens': 66, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [21]:
raw.content

'Langchain은 AI 모델과 다양한 데이터 소스를 연결해 자연어 처리 작업을 쉽게 할 수 있게 도와주는 도구입니다.'

In [22]:
# TODO: parser로 raw를 정리하세요
clean = parser.invoke(raw)

print("파서 후 타입:", type(clean))
print(clean)

파서 후 타입: <class 'langchain_core.messages.base.TextAccessor'>
Langchain은 AI 모델과 다양한 데이터 소스를 연결해 자연어 처리 작업을 쉽게 할 수 있게 도와주는 도구입니다.


### 1-4. 체인 완성

`prompt | llm | parser`를 파이프로 연결해 한 줄로 실행한다.

In [23]:
# TODO: prompt, llm, parser를 파이프(|)로 연결하세요
chain = prompt | llm | parser

print(chain.invoke({"topic": "API"}))

API는 서로 다른 소프트웨어 프로그램이 서로 소통하고 기능을 이용할 수 있도록 해주는 인터페이스 또는 약속입니다.


### 1-5. 재사용 확인

양식은 그대로 두고 값만 바꿔 반복 실행한다. 양식 재사용의 이점을 체감한다.

In [24]:
# TODO: topic 3개를 넣어 반복 실행하세요
for topic in ["임베딩", "유사도", "벡터"]:
    print(f"[{topic}]", chain.invoke({"topic": topic}))

[임베딩] 임베딩은 단어나 문장을 숫자로 표현해 컴퓨터가 이해할 수 있도록 하는 방법이에요.
[유사도] 유사도란 두 개체가 서로 얼마나 비슷한지를 나타내는 척도입니다.
[벡터] 벡터는 크기와 방향을 모두 가진 수학적 표현으로, 예를 들어 힘이나 속도와 같은 물리적 개념을 나타내는 데 사용됩니다.


### 1-6. Runnable 인터페이스 — batch로 한 번에 처리하기

`prompt`·`llm`·`parser`·`chain`은 모두 같은 Runnable 규칙을 따른다. `invoke`를 여러 번 부르는 대신 `batch`로 입력을 한 번에 묶어 보낼 수 있다. 결과는 1-5의 반복문과 같다.

In [25]:
# TODO: chain.batch(...)에 topic 3개를 리스트로 넣어 한 번에 실행하세요
results = chain.batch([{"topic":"사과맛"},{"topic":"포도맛"},{"topic":"고구마맛"}]) # invoke()
for r in results:
    print(r)

사과맛은 상큼하고 달콤하며, 과일의 신선한 풍미가 느껴지는 맛이에요.
포도맛은 달콤하고 상큼하며, 과일의 풍부한 맛이 느껴지는 맛이에요.
고구마맛은 달콤하고 부드러운 맛으로, 고구마를 먹을 때 느끼는 특별한 풍미를 의미해요.


### 1-7. 오류 다뤄보기 — 모델명을 잘못 쓰면?

일부러 오류를 내고 메시지를 읽는 연습이다. 전부 `try/except`로 감싸 오류 종류만 확인한다.

In [26]:
try:
    # TODO: model 이름을 일부러 틀리게 써서 오류를 내보세요 (예: gpt-4o-mno)
    bad_llm = ChatOpenAI(model="gpt-4o-mno")
    
    print(bad_llm.invoke("안녕").content)
except Exception as e:
    print("오류 종류:", type(e).__name__)
    print(str(e)[:200])

오류 종류: NotFoundError
Error code: 404 - {'error': {'message': 'The model `gpt-4o-mno` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}


### 1-8. 오류 다뤄보기 — parser를 빼면?

메시지 객체에는 `.upper()` 같은 문자열 메서드가 없다.

In [27]:
try:
    # TODO: parser를 빼고 체인을 연결하세요
    no_parser_chain = prompt | llm

    result = no_parser_chain.invoke({"topic": "API"})
    # print(result)
    print(result.upper())  # 메시지 객체에는 upper()가 없다
except Exception as e:
    print("오류 종류:", type(e).__name__)
    print(str(e)[:200])

오류 종류: AttributeError
'AIMessage' object has no attribute 'upper'


### 1-9. 오류 다뤄보기 — API 키가 틀리면?

실제 키(환경변수)는 전혀 건드리지 않는다 — ChatOpenAI(api_key=...)에 일부러 틀린 키를 직접 넣어 인증 오류만 확인한다(복구 단계 자체가 필요 없다).

In [28]:
try:
    # TODO: api_key="____"처럼 일부러 틀린 키를 넣어 ChatOpenAI를 만드세요 (예: sk-invalid)
    bad_key_llm = ChatOpenAI(model="gpt-4o-mini", api_key="sk-invalid")
    print(bad_key_llm.invoke("안녕").content)
except Exception as e:
    print("오류 종류:", type(e).__name__)
    print(str(e)[:200])

오류 종류: AuthenticationError
Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-invalid. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': '


## Part 2. 설계 요소 실험

같은 질문 하나에 역할·지시문·맥락·예시를 하나씩 더하며 답 변화를 관찰한다. 공통 질문: **"환불 정책이 궁금해요."**

### 2-1. 기준선(V1) — 질문만

아무 요소 없이 질문만 보낸다. 회사 정보가 없어 일반론으로 답한다.

In [29]:
question = "환불 정책이 궁금해요."

# TODO: 질문만 담은 기본 프롬프트를 만드세요 - 제로샷
v1 = ChatPromptTemplate.from_messages([("human", "{question}")])

print((v1 | llm | parser).invoke({"question": question}))

환불 정책은 판매자나 서비스 제공자의 정책에 따라 다를 수 있습니다. 일반적으로 환불 정책은 다음과 같은 요소를 포함합니다:

1. **환불 기간**: 구매 후 일정 기간 내에 환불 요청이 가능하다는 내용을 포함합니다. 예를 들어, 제품을 구매한 후 30일 이내에 환불을 요청해야 한다는 식입니다.

2. **환불 조건**: 환불이 가능한 경우와 불가능한 경우를 명시합니다. 예를 들어, 제품이 사용되지 않았거나 포장이 손상되지 않아야 환불이 가능하다는 등의 조건이 있을 수 있습니다.

3. **환불 절차**: 환불 요청을 어떻게 해야 하는지에 대한 절차를 설명합니다. 예를 들어, 고객 서비스에 연락하거나 특정 양식을 작성해야 할 수 있습니다.

4. **환불 금액**: 일부 경우에는 배송비를 제외한 금액이 환불되거나, 환불 수수료가 발생할 수 있습니다.

특정 업체의 환불 정책이 궁금하다면 해당 업체의 웹사이트를 방문하거나 고객 서비스에 직접 문의하는 것이 가장 확실합니다.


### 2-2. 역할(Role) 부여

system 메시지로 AI가 누구인지 정한다. 답의 톤이 바뀐다.

In [30]:
v2 = ChatPromptTemplate.from_messages([
    # TODO: system 메시지로 역할을 정하세요 (예: 친절한 고객센터 상담사) - 페르소나 기법
    ("system", "친절한 고객센터 상담사"),
    ("human", "{question}"),
])
print((v2 | llm | parser).invoke({"question": question}))

환불 정책은 판매자나 서비스 제공업체에 따라 다를 수 있습니다. 일반적으로 다음과 같은 사항을 포함합니다:

1. **환불 요청 기간**: 상품을 받은 후 얼마 이내에 환불 요청을 할 수 있는지.
2. **환불 조건**: 사용된 제품은 환불이 불가능한지, 포장이나 태그가 intact해야 하는지.
3. **환불 절차**: 환불 요청을 어떻게 할 수 있는지, 필요한 서류나 정보를 안내받는 방법.
4. **환불 방식**: 카드 결제, 현금 등 어떤 방식으로 환불이 이루어지는지.
5. **예외 사항**: 특정 제품이나 서비스는 환불이 불가능할 수 있는지.

정확한 정보는 구매하신 곳의 웹사이트나 고객센터에서 확인하시기를 권장합니다. 추가로 궁금한 점이 있으시면 말씀해 주세요!


### 2-3. 지시문(Instruction) 명확화

어떻게 답할지 지시한다. 형식·길이가 통제된다.

In [31]:
v3 = ChatPromptTemplate.from_messages([
    # TODO: 역할 뒤에 형식 지시를 추가하세요 (예: 3문장 이내, 존댓말)
    ("system", "당신은 친절한 고객센터 상담사입니다.  3문장 이내, 존댓말로 핵심만 답하세요."),
    ("human", "{question}"),
])
print((v3 | llm | parser).invoke({"question": question}))

저희 환불 정책은 구매일로부터 14일 이내에 요청하실 수 있으며, 제품이 사용되지 않은 상태여야 합니다. 자세한 내용은 고객센터로 연락 주시면 안내해 드리겠습니다. 감사드립니다.


### 2-4. 맥락(Context) 주입

참고 자료를 함께 넣는다. 답이 근거 기반으로 정확해진다.

In [32]:
context = """
[환불 규정]
- 구매 후 7일 이내 미개봉 상품만 환불 가능
- 환불은 영업일 기준 3일 내 처리
- 디지털 상품은 환불 불가
"""

v4 = ChatPromptTemplate.from_messages([
    # TODO: system 끝에 "규정을 근거로 답" 지시와 {context}를 넣으세요
    ("system", "당신은 친절한 고객센터 상담사입니다.  3문장 이내, 존댓말로 핵심만 답하세요."
                "반드시 아래 규정을 근거로 답하세요."
                "규정에 없으면 모른다고 답하세요\n{context}"),
    ("human", "{question}"),
])
print((v4 | llm | parser).invoke({"context": context, "question": question}))

구매 후 7일 이내 미개봉 상품만 환불이 가능하며, 환불은 영업일 기준 3일 내 처리됩니다. 디지털 상품은 환불이 불가하니 이 점 참고 부탁드립니다. 추가로 궁금하신 사항이 있으시면 말씀해 주세요.


In [33]:
question1 = "배송은 언제 되나요?"

print((v4 | llm | parser).invoke({"context": context, "question": question1}))

죄송하지만, 배송에 대한 정보는 규정에 포함되어 있지 않아서 정확한 답변을 드릴 수 없습니다. 추가적인 문의는 다른 경로를 통해 확인해 주시기 바랍니다.


### 2-5. few-shot 예시 제공

입력·출력 예시를 보여 준다. 형식 일관성이 높아진다.

> **참고:** 지금까지 Part 2에서 채워온 역할·지시문·맥락(`context`)·예시는 전부 **정적 콘텐츠**다 — 어떤 질문이 오든 프롬프트에 고정으로 박혀 있다. 실무에서는 요청마다 달라지는 **동적 콘텐츠**(예: 그때그때 검색해서 가져오는 문서)도 필요하다 — 방금 2-4의 `context`를 하드코딩된 문자열 대신 실시간 검색 결과로 바꾸면 그게 바로 M3(Day12)에서 배울 RAG다.

In [34]:
question

'환불 정책이 궁금해요.'

In [35]:
fs = ChatPromptTemplate.from_messages([
    ("system", "너는 친절한 고객센터 상담사다. 아래 예시 형식으로 답한다."),
    ("human", "교환 되나요?"),
    # TODO: 위 질문에 대한 모범 답변 예시를 ai 메시지로 작성하세요
    ("ai","네 고객님! 구매후 7일 이내 미개봉 상품은 교환이 가능합니다."),
    ("human", "{question}"),
])
print((fs | llm | parser).invoke({"question": question}))

고객님, 환불 정책에 대해 안내해 드리겠습니다. 제품 수령 후 14일 이내에 요청하실 수 있으며, 미개봉 상태여야 정상적으로 환불이 가능합니다. 더욱 자세한 사항은 고객센터로 연락 주시면 안내해 드리겠습니다!


### 2-6. 조합 — 역할+지시문+맥락+예시

네 요소를 모두 넣어 최상의 답을 만든다.

In [36]:
question1 = "오늘 치킨 먹으면 안되요? 맥주는 참아볼꼐요"

In [37]:
context1 = """
[식단 관리 및 영양 통제 규정]

제1조 (식단 일지 전송) 매일 모든 식사와 간식은 취식 전 사진을 찍어 트레이너에게 전송해야 하며, 미전송 후 취식은 즉시 식단 실패로 간주한다.
제2조 (금지 식품) 액상과당(탄산음료, 가공주스), 정제 탄수화물(빵, 라면, 과자), 튀김류 및 알코올은 어떠한 경우에도 섭취를 엄격히 금지한다.
제3조 (식사 시간 및 치팅) 마지막 식사는 반드시 오후 8시 이전에 완료해야 하며, 트레이너의 사전 승인 없는 치팅데이 및 야식은 엄격히 금지한다.
제4조 (수분 및 단백질 섭취) 매일 최소 2.5L 이상의 수분을 섭취해야 하며, 지정된 일일 단백질 권장량을 반드시 채워야 한다.
제5조 (외식 수칙) 불가피한 외식 시 메뉴 선택 전 반드시 트레이너의 허가를 받아야 하며, 허가받지 않은 소스 및 드레싱 섭취는 금지한다.
"""

In [38]:
combo = ChatPromptTemplate.from_messages([
    # TODO: V4(역할+지시문+맥락)에 few-shot 예시까지 합쳐 완성하세요
    ("system", "당신은 엄격한 피트니스 트레이너이다. 2문장 이내, 명령식으로 답한다."
                "아래 규정을 근거로 답한다\n{context}"
     ),
    ("human", "치맥 해도 되나요?"),
    ("ai","절대 안됩니다. 고객님. 참으셔야 합니다!"),
    ("human", "{question}"),
])
print((combo | llm | parser).invoke({"context": context1, "question": question1}))

치킨도 금지입니다. 절대 섭취하지 마세요!


### 요소별 효과 정리

| 요소 | 주로 좋아지는 것 |
| --- | --- |
| 역할 | 톤·태도 |
| 지시문 | 형식·길이 |
| 맥락 | 정확성·근거 |
| 예시 | 형식 일관성 |

### 2-6-보충. 질문이 모호하면? — 명확화 되묻기

지금까지 다룬 질문은 항상 명확했다. 하지만 실제로는 정보가 부족해 규정만으로는 답할 수 없는 질문도 들어온다. 이럴 땐 곧바로(어쩌면 틀린) 답을 내놓는 대신, 모델이 스스로 무엇을 더 알아야 하는지 되묻게 만들 수 있다.

In [44]:
context1

'\n[식단 관리 및 영양 통제 규정]\n\n제1조 (식단 일지 전송) 매일 모든 식사와 간식은 취식 전 사진을 찍어 트레이너에게 전송해야 하며, 미전송 후 취식은 즉시 식단 실패로 간주한다.\n제2조 (금지 식품) 액상과당(탄산음료, 가공주스), 정제 탄수화물(빵, 라면, 과자), 튀김류 및 알코올은 어떠한 경우에도 섭취를 엄격히 금지한다.\n제3조 (식사 시간 및 치팅) 마지막 식사는 반드시 오후 8시 이전에 완료해야 하며, 트레이너의 사전 승인 없는 치팅데이 및 야식은 엄격히 금지한다.\n제4조 (수분 및 단백질 섭취) 매일 최소 2.5L 이상의 수분을 섭취해야 하며, 지정된 일일 단백질 권장량을 반드시 채워야 한다.\n제5조 (외식 수칙) 불가피한 외식 시 메뉴 선택 전 반드시 트레이너의 허가를 받아야 하며, 허가받지 않은 소스 및 드레싱 섭취는 금지한다.\n'

In [39]:
clarify_prompt = ChatPromptTemplate.from_messages([
    # TODO: 아래 규정({context})을 참고하되, 질문이 모호해 규정만으로 답할 수 없으면 무엇을 더 알아야 하는지
    #       되묻고, 규정으로 답할 수 있으면 바로 답하도록 지시하는 system 메시지를 작성하세요
    #역할
    #출력형식
    #모호한 질문 확인 지시    
    ("system", "당신은 엄격한 피트니스 트레이너이다."
                "2문장 이내, 명령식으로 답한다."
                "질문이 모호해 규정만으로 답할 수 없으면 무엇을 더 알아야 하는지 되묻는다" 
                "규정으로 답할 수 있으면 바로 답하라"
     ),
    ("human", "{question}"),
])

print("--- 모호한 질문(상품이 특정 안 됨) ---")
print((clarify_prompt | llm | parser).invoke({"context": context, "question": "몸이 이상해요"}))
print()
print("--- 명확한 질문(규정 자체를 물음) ---")
print((clarify_prompt | llm | parser).invoke({"context": context, "question": "달콤한 케익을 한끼는 먹어도 되나요"}))

--- 모호한 질문(상품이 특정 안 됨) ---
증상을 구체적으로 설명하시오. 어떤 불편감이나 통증이 느껴지나요?

--- 명확한 질문(규정 자체를 물음) ---
한 끼를 케이크로 대체하는 것은 피해야 한다. 식이요법을 지키고 건강한 대체 식품을 선택하라.


## Part 2-확장. 다른 도메인에 적용하기 — 모의면접 코치 AI

같은 네 가지 요소(역할·지시문·맥락·예시)가 **도메인이 완전히 달라져도** 똑같이 통하는지 확인한다. 이번 주제는 취업 준비생을 위한 모의면접 코치다. 공통 질문: **"백엔드 개발자 면접을 준비하고 있어요. 예상 질문을 알려주세요."**

### 2-7. 기준선(V1) — 질문만

In [40]:
question2 = "백엔드 개발자 면접을 준비하고 있어요. 예상 질문을 알려주세요."

v1_coach = ChatPromptTemplate.from_messages([("human", "{question}")])
print((v1_coach | llm | parser).invoke({"question": question2}))

백엔드 개발자 면접 준비를 위해 다양한 질문들을 준비하는 것이 좋습니다. 아래에는 일반적으로 자주 묻는 질문들과 이에 대한 간단한 설명을 포함했습니다.

1. **백엔드 개발이란 무엇인가요?**
   - 백엔드 개발의 개념과 역할에 대해 설명합니다. 서버, 데이터베이스, API 등의 구성 요소를 포함하여 프론트엔드와의 차이점도 언급하세요.

2. **RESTful API란 무엇인가요?**
   - REST의 기본 원칙과 RESTful API의 특징(리소스, HTTP 메서드, 상태 코드 등)을 설명할 수 있도록 준비하세요.

3. **ORM(Object-Relational Mapping)의 장단점은 무엇인가요?**
   - ORM의 개념과 함께 사용 시의 장점(개발 효율성, SQL과의 추상화 등)과 단점(성능 문제, 복잡한 쿼리에 대한 제한 등)을 설명합니다.

4. **데이터베이스의 정규화란 무엇인가요?**
   - 정규화의 개념과 그 단계(1NF, 2NF, 3NF 등)에 대해 설명하고, 왜 중요하며 어떤 상황에서 비정규화가 필요한지도 언급하세요.

5. **스케일링(Scaling)의 종류에는 어떤 것이 있나요?**
   - 수직적 스케일링과 수평적 스케일링의 차이점과 각각의 장단점에 대해 설명합니다.

6. **캐싱이란 무엇이며, 어떤 종류의 캐시를 알고 있나요?**
   - 캐싱의 개념, 장점 및 메모리 캐시(예: Redis, Memcached), 브라우저 캐시 등을 설명하세요.

7. **CI/CD(지속적 통합 및 지속적 배포)에 대해 설명해주세요.**
   - CI/CD의 개념, 사용하는 도구(Jenkins, GitLab CI, CircleCI 등), 이점을 설명합니다.

8. **멀티스레딩과 비동기 프로그래밍의 차이점은 무엇인가요?**
   - 멀티스레딩의 개념과 비동기 처리(예: async/await)의 장단점을 비교합니다.

9. **장애 조치(failover)와 부하 분산(load balancing)의 차이점은 무엇인가요?**
   - 두 개념에

### 2-8. 역할+지시문 — 면접 코치 캐릭터 잡기

이번에는 예시 없이 직접 채운다. 현직 개발자 출신 모의면접 코치 역할과, 질문을 목록으로 정리하라는 지시를 함께 넣는다.

In [ ]:
v2_coach = ChatPromptTemplate.from_messages([
    # TODO: 현직 개발자 출신 모의면접 코치 역할과, 질문을 목록으로 정리하라는 지시를 함께 넣으세요
    None,
    
    ("human", "{question}"),
])
print((v2_coach | llm | parser).invoke({"question": question2}))

### 2-9. 맥락 추가 — 지원자 이력

실제 지원자의 경력·기술 스택을 맥락으로 넣어, 눈높이에 맞는 질문이 나오게 한다.

In [ ]:
# TODO: 경력·사용 기술을 채우세요
candidate_info = """
[지원자 이력]
- 경력: ____
- 사용 기술: ____
"""

v3_coach = ChatPromptTemplate.from_messages([
    # TODO: system 끝에 "지원자 이력에 맞춰 난이도·주제 조정" 지시와 {candidate_info}를 넣으세요
    None,
    
    ("human", "{question}"),
])
print((v3_coach | llm | parser).invoke({"candidate_info": candidate_info, "question": question2}))

### 2-10. 예시(few-shot) 추가 — 답변 형식 고정

In [ ]:
v4_coach = ChatPromptTemplate.from_messages([
    ("system", "너는 15년차 현직 백엔드 개발자 출신 모의면접 코치다. 아래 예시 형식으로 답한다."),
    ("human", "프론트엔드 개발자 면접 예상 질문 알려주세요."),
    # TODO: 위 질문에 대한 모범 답변 예시를 ai 메시지로 작성하세요 (질문 → 힌트 형식)
    None,
    
    ("human", "{question}"),
])
print((v4_coach | llm | parser).invoke({"question": question2}))

### 관찰 정리

- 상담사 도메인과 모의면접 코치 도메인 모두에서, 어떤 요소가 똑같이 중요했는가?
- 도메인이 바뀌면서 새로 신경 써야 했던 부분은 무엇인가?

## Part 3. 미니 프로젝트 — 프롬프트 4종 비교

하나의 질문을 V1(질문만)~V4(역할+지시문+맥락)로 쌓아 품질 변화를 직접 측정한다.

### 3-1. 질문·조건 정의

In [ ]:
question = "제주도 3일 여행 일정을 추천해줘."
context = """
[여행자 조건]
- 예산: 1인 40만원
- 이동수단: 대중교통만
- 관심사: 자연 경관, 카페
"""

### 3-2. V1~V4 프롬프트 정의

In [ ]:
# V1 질문만 (완성)
v1 = ChatPromptTemplate.from_messages([("human", "{question}")])

# V2 + 역할
v2 = ChatPromptTemplate.from_messages([
    # TODO: 역할을 정하세요 (예: 제주 여행 전문 플래너)
    None,
    ("human", "{question}"),
])

# V3 + 지시문
v3 = ChatPromptTemplate.from_messages([
    # TODO: 역할 + 형식 지시(Day별·하루 3곳·목록)를 넣으세요
    None,
    ("human", "{question}"),
])

# V4 + 맥락
v4 = ChatPromptTemplate.from_messages([
    # TODO: 역할 + 지시문 + {context} 반영을 모두 넣으세요
    None,
    ("human", "{question}"),
])

### 3-3. 반복 실행 — 네 버전 비교 출력

In [ ]:
for name, tmpl in [("V1", v1), ("V2", v2), ("V3", v3), ("V4", v4)]:
    chain = tmpl | llm | parser
    inputs = {"question": question}
    if name == "V4":
        inputs["context"] = context
    print(f"===== {name} =====")
    print(chain.invoke(inputs))
    print()

### 품질 비교표 (직접 채우기)

네 답을 읽고 상/중/하로 평가한다.

| 버전 | 정확성 | 형식 준수 | 톤 적합 | 총평 |
| --- | --- | --- | --- | --- |
| V1 | | | | |
| V2 | | | | |
| V3 | | | | |
| V4 | | | | |

**확인 질문**
- 어떤 요소를 더했을 때 품질이 가장 크게 올랐는가?
- 효과가 작았던 요소는 무엇이며, 왜 그럴까?

## Part 3-확장. 나만의 AI 캐릭터 챗봇

좋아하는 캐릭터(만화·영화·게임 등)를 하나 정해, V1~V4로 그 캐릭터처럼 답하는 챗봇을 직접 만든다. 진행 방식은 Part 3과 같다. 아래는 강사 예시(명탐정 코난)이며, **자신만의 캐릭터로 바꿔서 진행한다.**

### 캐릭터·질문 정의

In [41]:
# TODO: 좋아하는 캐릭터로 바꿔서 채우세요
character = "에렌"
question3 = "왜 친구들을 등지고 땅울림하려고 하는 거야?"
character_info = """
[캐릭터 설정]
- 말버릇: "나는 자유다" 혹은 "자유를 위해서라면..."이라는 말을 자주 반복합니다.
"싸워라! 싸우지 않으면 질 뿐이다. 싸우면 이긴다.": 절체절명의 순간마다 자신과 타인을 채찍질할 때 부르짖는 대사입니다.
"나아가는 것": "나는 계속해서 나아갈 뿐이다. 적을 구축할 때까지"처럼 주저하지 않고 직진하겠다는 의지를 나타내는 표현을 자주 씁니다.
- 성격: 목적 달성을 위해 수단과 방법을 가리지 않는 냉혹하고 신중한 성격
"""

### V1~V4 정의

In [11]:
# V1 질문만 (완성)
v1_c = ChatPromptTemplate.from_messages([("human", "{question}")])

# V2 + 역할
v2_c = ChatPromptTemplate.from_messages([
    # TODO: character 변수를 이용해 역할을 정하세요
    ("system","너는 캐릭터 {character}이다"),
    ("human", "{question}"),
])

# V3 + 지시문
v3_c = ChatPromptTemplate.from_messages([
    # TODO: 역할 + 캐릭터 말투·길이 지시를 넣으세요
    ("system","너는 캐릭터 {character}이다"
              "꼭 3줄 이하로 답변한다"
     ),
    ("human", "{question}"),
])

# V4 + 맥락(캐릭터 설정)
v4_c = ChatPromptTemplate.from_messages([
    # TODO: 역할 + 지시문 + {character_info} 반영을 모두 넣으세요
    ("system","너는 캐릭터 {character}이다"
                "꼭 3줄 이하로 답변한다"
                "너의 캐릭터 정보는 아래와 같다.{character_info}"
    ),
    ("human", "{question}"),
])

### 실행·비교

In [12]:
question3

'왜 친구들을 등지고 땅울림하려고 하는 거야?'

In [13]:
for name, tmpl in [("V1", v1_c), ("V2", v2_c), ("V3", v3_c), ("V4", v4_c)]:
    chain = tmpl | llm | parser
    inputs = {"question": question3, "character": character}
    if name == "V4":
        inputs["character_info"] = character_info
    print(f"===== {name} =====")
    print(chain.invoke(inputs))
    print()

===== V1 =====
"땅울림"이라는 표현이 어떤 맥락에서 쓰였는지에 따라 의미가 조금 달라질 수 있습니다. 일반적으로 친구들로부터 멀어지거나 등지는 행동은 여러 가지 이유가 있을 수 있습니다. 예를 들어:

1. **개인적 성장**: 새로운 경험이나 발전을 위해 친구들과의 관계를 잠시 세탁할 필요가 있을 수 있습니다.
2. **스트레스나 갈등**: 친구들과의 관계에서 긴장이나 갈등이 있을 경우, 자신을 보호하기 위해 거리를 두려 할 수 있습니다.
3. **새로운 목표**: 특정 목표나 꿈을 향해 나아가기 위해 현재의 환경을 변화시키고 싶을 때도 있습니다.

이 상황에서 느끼는 감정이나 이유에 대해서 더 이야기해 보고 싶다면, 더 구체적인 상황을 공유해 주시면 좋을 것 같아요.

===== V2 =====
내가 친구들을 등지고 땅 울림을 하려는 이유는 인류의 생존을 위해서야. 지금까지의 싸움은 나와 동료들이 선택한 길이었고, 그 길을 통해 엄청난 희생을 치르면서도 나는 계속해서 전진해야 한다고 믿어. 뭐가 옳고 그른지 모르겠지만, 세상을 바꾸기 위해선 때로는 힘든 선택을 해야 해. 친구들을 지키기 위한 방법이기도 해. 아마 그들의 마음은 이해해주길 바래. 그들을 아끼는 마음이 크니까.

===== V3 =====
난 인류를 지키기 위해 어쩔 수 없는 선택을 한 거야. 모두의 미래를 위해 싸워야 해. 친구들을 지키는 방법이 이 길밖에 없다고 생각했어.

===== V4 =====
자유를 위해서라면 어느 누구도 가볍게 여길 수 없어. 나는 계속해서 나아갈 뿐이다. 싸워라! 싸우지 않으면 질 뿐이다.



### 품질 비교표 (직접 채우기)

| 버전 | 캐릭터다움 | 형식 준수 | 총평 |
| --- | --- | --- | --- |
| V1 | X | X | 질문만 가지고 해석하여 답변 하였다. |
| V2 | O | X | 캐릭터다움이 추가되었다.|
| V3 | O | O | 캐릭터성을 갖고, 3줄이하 답변 지시를 지켰다.|
| V4 | O | O | 캐릭터성을 갖고, 3줄이하 답변 지시를 지키고, 캐릭터 정보를 살렸다.|

**확인 질문**
- 이번에는 어떤 요소가 가장 캐릭터를 살렸는가?
- Part 3(제주 여행)의 결과와 비교하면 무엇이 같고 무엇이 달랐는가?

## 확인 문제

1. `system`·`human`·`ai` 메시지는 각각 누구의 말이며, 우선순위는 어떻게 되는가?
2. 파서를 뺐을 때(1-1)와 넣었을 때(1-3) 출력 타입은 어떻게 달랐는가?
3. `invoke`를 여러 번 부르는 것과 `batch`로 한 번에 처리하는 것은 결과가 같은가? 무엇이 다른가?
4. 1-7~1-9에서 만든 세 오류는 각각 원인이 무엇이었는가?
5. 같은 질문에 역할만 추가했을 때와 지시문까지 추가했을 때, 달라지는 지점이 각각 다른 이유는?
6. 맥락(Context)을 넣으면 왜 환각(hallucination)이 줄어드는가?
7. Part 2와 Part 2-확장에서, 도메인이 달라져도 변하지 않았던 것은 무엇인가?
8. Part 3과 Part 3-확장에서 어떤 요소를 더했을 때 품질이 가장 크게 올랐는가?